# Organizando a saída por Copa

## Task 2: organizar a saída por Copa

O painel pergunta por **uma Copa** de cada vez: a saída fica organizada **por Copa**.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-particionamento")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
EVENTOS = "../../data/events"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

## Duas coisas com o mesmo nome

| | Partição **em memória** | Partição **em disco** |
|---|---|---|
| O que é | um pedaço do dado, processado por uma task | uma pasta com os arquivos de um valor de coluna |
| Quem controla | `repartition`, `coalesce`, `shuffle.partitions` | `partitionBy`, na escrita |

## partitionBy: a coluna vira pasta

O `partitionBy` grava uma pasta por valor, como `ano=2018`. Na leitura, o nome da pasta volta a ser coluna:

## Partition pruning: não abrir o que não precisa

Procure **`PartitionFilters`** no plano: com um filtro na coluna de partição, o Spark só abre as pastas que precisa
(**partition pruning**).

Fora da partição, o filtro vai para **`PushedFilters`**: o Spark abre todas as pastas.

**Como escolher a coluna:** a que mais aparece nos filtros, com **poucos valores diferentes** (ano, país; não um
identificador).

## O problema dos arquivos pequenos

**Cada task grava um arquivo em cada pasta que recebe dados.** O `inputFiles()` lista os arquivos da versão atual
da tabela Delta.

Uma task, um arquivo em cada pasta. Simulando 20 tasks com `repartition(20)`:

**Cada arquivo tem um custo fixo** para abrir: centenas de arquivos pequenos (**small files**) pagam esse custo
centenas de vezes.

## Resultado da task 2

Saída por Copa, com partition pruning. Próximo vídeo: os arquivos pequenos e a Copa de 2026.

## Para consultar depois

- **Mais de uma coluna:** `partitionBy("ano", "fase")` cria pastas dentro de pastas (`ano=2018/fase=Final`). Cuidado:
  o número de pastas é a combinação dos valores, e cada pasta pode ficar com pouco dado.
- **`maxRecordsPerFile`:** `.option("maxRecordsPerFile", 50000)` limita as linhas por arquivo e quebra só as pastas
  grandes.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()